# Attribution Analysis by Channel

Analyze acquisition and conversion behavior by marketing channel to understand how each channel contributes across the funnel. The notebook focuses on contribution share, conversion quality, and **possible** over-attribution or under-attribution patterns using channel-level gaps between reach, clicks, conversions, spend, and return.


## What This Notebook Answers

- Which channels dominate impressions, clicks, conversions, spend, and attributed return?
- Which channels attract traffic efficiently, and which convert that traffic efficiently?
- Which channels appear **over-attributed** under click-centric thinking?
- Which channels appear **under-attributed** because they convert or return value better than their click share suggests?
- What should change in budget allocation and attribution interpretation?


## Dataset And Attribution Guardrail

**Dataset:** public multi-channel marketing workbook from Ayeni's campaign analysis repository.

Available fields include:

- `Channel_Used`
- `Impressions`
- `Clicks`
- `Conversion_Rate`
- `Acquisition_Cost`
- `ROI`
- audience and campaign descriptors

Important limitation:

- This is **not** a true path-level attribution dataset.
- There are no user journeys across multiple channels, no touch sequences, and no assisted-conversion paths.
- Therefore, the notebook identifies **possible over-attribution and under-attribution patterns** using contribution-share mismatches:
  - channels with larger click share than conversion / return share can look stronger than they really are in click-heavy reporting
  - channels with smaller click share but larger conversion / return share can be under-credited in click-heavy reporting


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_URL = "https://raw.githubusercontent.com/Ayeni01/Market-Campaign-Analysis/main/marketing_campaign_dataset.xlsx"
print(f"Data URL: {DATA_URL}")


Data URL: https://raw.githubusercontent.com/Ayeni01/Market-Campaign-Analysis/main/marketing_campaign_dataset.xlsx


## Load And Inspect The Data

Load the workbook and confirm the available channels, scale, and the raw metrics needed to build contribution and attribution-gap views.


In [2]:
df_raw = pd.read_excel(DATA_URL)

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Channels: {sorted(df_raw['Channel_Used'].unique().tolist())}")
display(df_raw.head())
display(df_raw.sample(5, random_state=42))


Shape: 200,005 rows x 15 columns
Channels: ['Email', 'Facebook', 'Google Ads', 'Instagram', 'Website', 'YouTube']


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Date,Clicks,Impressions,Engagement_Score,Customer_Segment
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.0400,16174,6.2900,Chicago,44197,506,1922,6,Health & Wellness
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.1200,11566,5.6100,New York,44228,116,7523,7,Fashionistas
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.0700,10200,7.1800,Los Angeles,44256,584,7698,1,Outdoor Adventurers
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.1100,12724,5.5500,Miami,44287,217,1820,7,Health & Wellness
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.0500,16452,6.5000,Los Angeles,44317,379,4201,3,Health & Wellness


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Date,Clicks,Impressions,Engagement_Score,Customer_Segment
167321,167322,Alpha Innovations,Social Media,Women 35-44,45 days,Email,0.0300,18461,4.4400,Chicago,44202,631,9774,6,Foodies
24979,24980,TechCorp,Search,Women 25-34,45 days,Instagram,0.0500,8579,4.1600,Chicago,44445,709,6055,7,Health & Wellness
39676,39677,Alpha Innovations,Social Media,Men 25-34,45 days,Facebook,0.0200,19274,2.6400,New York,44453,920,9464,1,Foodies
81759,81760,TechCorp,Search,Women 25-34,45 days,Instagram,0.1500,19072,2.1900,New York,44561,187,7679,5,Outdoor Adventurers
120376,120377,Alpha Innovations,Search,All Ages,15 days,Google Ads,0.0900,6919,5.5100,Houston,44488,820,2097,6,Outdoor Adventurers


## Validation And KPI Engineering

Standardize names and derive consistent funnel metrics. Estimated conversions are calculated from `Clicks × Conversion_Rate`, because conversions are not stored as a separate raw field.


In [3]:
validation_report = pd.DataFrame(
    {
        "metric": ["rows", "columns", "missing channels", "missing clicks", "missing impressions", "missing conversion rate", "missing acquisition cost", "missing roi"],
        "value": [
            len(df_raw),
            df_raw.shape[1],
            int(df_raw['Channel_Used'].isna().sum()),
            int(df_raw['Clicks'].isna().sum()),
            int(df_raw['Impressions'].isna().sum()),
            int(df_raw['Conversion_Rate'].isna().sum()),
            int(df_raw['Acquisition_Cost'].isna().sum()),
            int(df_raw['ROI'].isna().sum()),
        ],
    }
)

df = df_raw.copy()
df.columns = [column.strip().lower() for column in df.columns]
df["date"] = pd.to_datetime(df["date"], unit="D", origin="1899-12-30")

numeric_cols = ["conversion_rate", "acquisition_cost", "roi", "clicks", "impressions", "engagement_score"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

df["conversions"] = (df["clicks"] * df["conversion_rate"]).round().astype(int)
df["ctr"] = df["clicks"] / df["impressions"].replace(0, np.nan)
df["cvr"] = df["conversions"] / df["clicks"].replace(0, np.nan)
df["purchase_rate"] = df["conversions"] / df["impressions"].replace(0, np.nan)
df["cpc"] = df["acquisition_cost"] / df["clicks"].replace(0, np.nan)
df["cpa"] = df["acquisition_cost"] / df["conversions"].replace(0, np.nan)
df["estimated_return"] = df["acquisition_cost"] * df["roi"]
df["estimated_revenue_proxy"] = df["acquisition_cost"] + df["estimated_return"]
df["month"] = df["date"].dt.to_period("M").astype(str)

display(validation_report)
display(df.head())


,metric,value
0,rows,200005
1,columns,15
2,missing channels,0
3,missing clicks,0
4,missing impressions,0
5,missing conversion rate,0
6,missing acquisition cost,0
7,missing roi,0


,campaign_id,company,campaign_type,target_audience,duration,channel_used,conversion_rate,acquisition_cost,roi,location,date,clicks,impressions,engagement_score,customer_segment,conversions,ctr,cvr,purchase_rate,cpc,cpa,estimated_return,estimated_revenue_proxy,month
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.0400,16174,6.2900,Chicago,2021-01-01,506,1922,6,Health & Wellness,20,0.2633,0.0395,0.0104,31.9644,808.7000,"101,734.4600","117,908.4600",2021-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.1200,11566,5.6100,New York,2021-02-01,116,7523,7,Fashionistas,14,0.0154,0.1207,0.0019,99.7069,826.1429,"64,885.2600","76,451.2600",2021-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.0700,10200,7.1800,Los Angeles,2021-03-01,584,7698,1,Outdoor Adventurers,41,0.0759,0.0702,0.0053,17.4658,248.7805,"73,236.0000","83,436.0000",2021-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.1100,12724,5.5500,Miami,2021-04-01,217,1820,7,Health & Wellness,24,0.1192,0.1106,0.0132,58.6359,530.1667,"70,618.2000","83,342.2000",2021-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.0500,16452,6.5000,Los Angeles,2021-05-01,379,4201,3,Health & Wellness,19,0.0902,0.0501,0.0045,43.4090,865.8947,"106,938.0000","123,390.0000",2021-05


## Overall Funnel Snapshot

Start with the global funnel before decomposing it by channel. This provides the baseline context for channel contribution shares.


In [4]:
overall = pd.DataFrame(
    {
        "metric": [
            "Impressions",
            "Clicks",
            "Estimated conversions",
            "Total acquisition cost",
            "Estimated return",
            "Average CTR",
            "Average CVR",
            "Average ROI",
        ],
        "value": [
            df["impressions"].sum(),
            df["clicks"].sum(),
            df["conversions"].sum(),
            df["acquisition_cost"].sum(),
            df["estimated_return"].sum(),
            df["clicks"].sum() / df["impressions"].sum(),
            df["conversions"].sum() / df["clicks"].sum(),
            df["roi"].mean(),
        ],
    }
)

display(overall)


,metric,value
0,Impressions,"1,101,488,958.0000"
1,Clicks,"109,957,667.0000"
2,Estimated conversions,"8,804,881.0000"
3,Total acquisition cost,"2,500,950,881.0000"
4,Estimated return,"12,517,681,491.8900"
5,Average CTR,0.0998
6,Average CVR,0.0801
7,Average ROI,5.0024


## Channel Contribution Table

Aggregate by channel and compute both volume KPIs and contribution shares. This is the central attribution table for the notebook.


In [5]:
channel_summary = (
    df.groupby("channel_used")
    .agg(
        campaigns=("channel_used", "size"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        spend=("acquisition_cost", "sum"),
        estimated_return=("estimated_return", "sum"),
        avg_roi=("roi", "mean"),
    )
    .reset_index()
)

channel_summary["ctr"] = channel_summary["clicks"] / channel_summary["impressions"]
channel_summary["cvr"] = channel_summary["conversions"] / channel_summary["clicks"]
channel_summary["purchase_rate"] = channel_summary["conversions"] / channel_summary["impressions"]
channel_summary["cpc"] = channel_summary["spend"] / channel_summary["clicks"]
channel_summary["cpa"] = channel_summary["spend"] / channel_summary["conversions"]
channel_summary["return_on_spend_proxy"] = channel_summary["estimated_return"] / channel_summary["spend"]

total_impressions = channel_summary["impressions"].sum()
total_clicks = channel_summary["clicks"].sum()
total_conversions = channel_summary["conversions"].sum()
total_spend = channel_summary["spend"].sum()
total_return = channel_summary["estimated_return"].sum()

channel_summary["impression_share"] = channel_summary["impressions"] / total_impressions
channel_summary["click_share"] = channel_summary["clicks"] / total_clicks
channel_summary["conversion_share"] = channel_summary["conversions"] / total_conversions
channel_summary["spend_share"] = channel_summary["spend"] / total_spend
channel_summary["return_share"] = channel_summary["estimated_return"] / total_return

display(channel_summary.sort_values("conversions", ascending=False).round(4))


,channel_used,campaigns,impressions,clicks,conversions,spend,estimated_return,avg_roi,ctr,cvr,purchase_rate,cpc,cpa,return_on_spend_proxy,impression_share,click_share,conversion_share,spend_share,return_share
0,Email,33599,184801107,18493963,1485425,420874104,"2,103,770,103.6800",4.9965,0.1001,0.0803,0.0080,22.7574,283.3358,4.9986,0.1678,0.1682,0.1687,0.1683,0.1681
4,Website,33361,183815901,18415351,1477696,416606897,"2,087,602,495.8100",5.0141,0.1002,0.0802,0.0080,22.6228,281.9300,5.0110,0.1669,0.1675,0.1678,0.1666,0.1668
2,Google Ads,33440,185020154,18342589,1468777,418944514,"2,097,955,331.9800",5.0031,0.0991,0.0801,0.0079,22.8400,285.2336,5.0077,0.1680,0.1668,0.1668,0.1675,0.1676
5,YouTube,33393,183450845,18350935,1463702,416797090,"2,084,397,748.7900",4.9937,0.1000,0.0798,0.0080,22.7126,284.7554,5.0010,0.1665,0.1669,0.1662,0.1667,0.1665
3,Instagram,33392,183738455,18316654,1462938,417124850,"2,079,976,528.7000",4.9887,0.0997,0.0799,0.0080,22.7730,285.1282,4.9865,0.1668,0.1666,0.1662,0.1668,0.1662
1,Facebook,32820,180662496,18038175,1446343,410603426,"2,063,979,282.9300",5.0187,0.0998,0.0802,0.0080,22.7630,283.8908,5.0267,0.1640,0.1640,0.1643,0.1642,0.1649


## Contribution Share Comparison

True channel contribution is easier to understand when all shares are shown side by side. Channels whose conversion or return share exceeds their click share often deserve more credit than click-centric reporting gives them.


In [6]:
share_plot = channel_summary.melt(
    id_vars=["channel_used"],
    value_vars=["impression_share", "click_share", "conversion_share", "spend_share", "return_share"],
    var_name="share_type",
    value_name="share",
)

plt.figure(figsize=(15, 7))
sns.barplot(data=share_plot, x="channel_used", y="share", hue="share_type")
plt.title("Channel Contribution Shares Across Funnel And Spend")
plt.xlabel("")
plt.ylabel("Share of total")
plt.xticks(rotation=20)
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_38584\1833729556.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conversion Behavior By Channel

Contribution alone is incomplete. This section checks whether each channel turns acquired attention into efficient conversion and return.


In [7]:
behavior_plot = channel_summary.melt(
    id_vars=["channel_used"],
    value_vars=["ctr", "cvr", "purchase_rate", "avg_roi"],
    var_name="metric",
    value_name="value",
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=behavior_plot[behavior_plot["metric"].isin(["ctr", "cvr", "purchase_rate"])], x="channel_used", y="value", hue="metric", ax=axes[0])
axes[0].set_title("Channel Conversion Behavior")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

efficiency = channel_summary[["channel_used", "cpc", "cpa", "return_on_spend_proxy"]].melt(
    id_vars=["channel_used"],
    var_name="metric",
    value_name="value",
)
sns.barplot(data=efficiency, x="channel_used", y="value", hue="metric", ax=axes[1])
axes[1].set_title("Channel Cost And Return Efficiency")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_38584\491131437.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Possible Over-Attribution And Under-Attribution Patterns

The next table operationalizes the attribution warning signs.

Definitions used here:

- **Possible over-attribution:** click share is materially larger than conversion share or return share
- **Possible under-attribution:** conversion share or return share is materially larger than click share

These are not path-level truth claims. They are channel contribution mismatches that signal where click-centric reporting may distort interpretation.


In [8]:
attribution = channel_summary.copy()
attribution["click_to_conversion_gap"] = attribution["click_share"] - attribution["conversion_share"]
attribution["click_to_return_gap"] = attribution["click_share"] - attribution["return_share"]
attribution["spend_to_return_gap"] = attribution["spend_share"] - attribution["return_share"]
attribution["undercredit_gap"] = attribution["conversion_share"] - attribution["click_share"]
attribution["return_undercredit_gap"] = attribution["return_share"] - attribution["click_share"]

attribution["possible_over_attribution_score"] = 100 * (
    0.45 * attribution["click_to_conversion_gap"].rank(pct=True)
    + 0.35 * attribution["click_to_return_gap"].rank(pct=True)
    + 0.20 * attribution["spend_to_return_gap"].rank(pct=True)
)
attribution["possible_under_attribution_score"] = 100 * (
    0.50 * attribution["undercredit_gap"].rank(pct=True)
    + 0.50 * attribution["return_undercredit_gap"].rank(pct=True)
)

over_view = attribution.sort_values("possible_over_attribution_score", ascending=False)[[
    "channel_used",
    "click_share",
    "conversion_share",
    "return_share",
    "spend_share",
    "click_to_conversion_gap",
    "click_to_return_gap",
    "spend_to_return_gap",
    "possible_over_attribution_score",
]]
under_view = attribution.sort_values("possible_under_attribution_score", ascending=False)[[
    "channel_used",
    "click_share",
    "conversion_share",
    "return_share",
    "undercredit_gap",
    "return_undercredit_gap",
    "possible_under_attribution_score",
]]

display(over_view.round(4))
display(under_view.round(4))

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=attribution,
    x="click_share",
    y="conversion_share",
    size="spend",
    hue="return_share",
    sizes=(200, 1500),
    palette="viridis",
)
max_share = max(attribution["click_share"].max(), attribution["conversion_share"].max())
plt.plot([0, max_share], [0, max_share], linestyle="--", color="gray")
for _, row in attribution.iterrows():
    plt.text(row["click_share"], row["conversion_share"], row["channel_used"], fontsize=10)
plt.title("Click Share vs Conversion Share By Channel")
plt.xlabel("Click share")
plt.ylabel("Conversion share")
plt.show()


,channel_used,click_share,conversion_share,return_share,spend_share,click_to_conversion_gap,click_to_return_gap,spend_to_return_gap,possible_over_attribution_score
3,Instagram,0.1666,0.1662,0.1662,0.1668,0.0004,0.0004,0.0006,86.6667
5,YouTube,0.1669,0.1662,0.1665,0.1667,0.0007,0.0004,0.0001,81.6667
4,Website,0.1675,0.1678,0.1668,0.1666,-0.0004,0.0007,-0.0002,56.6667
2,Google Ads,0.1668,0.1668,0.1676,0.1675,0.0000,-0.0008,-0.0001,51.6667
0,Email,0.1682,0.1687,0.1681,0.1683,-0.0005,0.0001,0.0002,41.6667
1,Facebook,0.1640,0.1643,0.1649,0.1642,-0.0002,-0.0008,-0.0007,31.6667


,channel_used,click_share,conversion_share,return_share,undercredit_gap,return_undercredit_gap,possible_under_attribution_score
0,Email,0.1682,0.1687,0.1681,0.0005,-0.0001,83.3333
1,Facebook,0.1640,0.1643,0.1649,0.0002,0.0008,83.3333
2,Google Ads,0.1668,0.1668,0.1676,-0.0000,0.0008,66.6667
4,Website,0.1675,0.1678,0.1668,0.0004,-0.0007,50.0000
3,Instagram,0.1666,0.1662,0.1662,-0.0004,-0.0004,33.3333
5,YouTube,0.1669,0.1662,0.1665,-0.0007,-0.0004,33.3333


C:\Users\ahmad\AppData\Local\Temp\ipykernel_38584\626068967.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Audience And Channel Interaction

Sometimes a channel looks weak overall because it is uneven across customer segments. This view checks whether some channels contribute disproportionately well inside specific audience segments.


In [9]:
segment_channel = (
    df.groupby(["customer_segment", "channel_used"])
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        spend=("acquisition_cost", "sum"),
        estimated_return=("estimated_return", "sum"),
    )
    .reset_index()
)
segment_channel["ctr"] = segment_channel["clicks"] / segment_channel["impressions"]
segment_channel["cvr"] = segment_channel["conversions"] / segment_channel["clicks"]
segment_channel["return_on_spend_proxy"] = segment_channel["estimated_return"] / segment_channel["spend"]

cvr_heatmap = segment_channel.pivot(index="customer_segment", columns="channel_used", values="cvr")
ros_heatmap = segment_channel.pivot(index="customer_segment", columns="channel_used", values="return_on_spend_proxy")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(cvr_heatmap, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("CVR By Customer Segment And Channel")

sns.heatmap(ros_heatmap, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Return-On-Spend Proxy By Segment And Channel")

plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_38584\2355770285.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Time Trend By Channel

Attribution patterns can drift over time. A channel that looks over-credited overall may still be important in specific months if its lower-funnel share strengthens in those periods.


In [10]:
monthly_channel = (
    df.groupby(["month", "channel_used"])
    .agg(
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        spend=("acquisition_cost", "sum"),
        estimated_return=("estimated_return", "sum"),
    )
    .reset_index()
)
monthly_channel["cvr"] = monthly_channel["conversions"] / monthly_channel["clicks"]
monthly_channel["return_on_spend_proxy"] = monthly_channel["estimated_return"] / monthly_channel["spend"]

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
sns.lineplot(data=monthly_channel, x="month", y="cvr", hue="channel_used", marker="o", ax=axes[0])
axes[0].set_title("Monthly CVR By Channel")
axes[0].tick_params(axis="x", rotation=45)

sns.lineplot(data=monthly_channel, x="month", y="return_on_spend_proxy", hue="channel_used", marker="o", ax=axes[1])
axes[1].set_title("Monthly Return-On-Spend Proxy By Channel")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_38584\3057811299.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Statistical Checks

The tests below ask whether channel-level conversion-rate differences are plausibly distinguishable and whether click share tends to move with return share in a reliable way.


In [11]:
channel_groups = []
for _, group in df.groupby("channel_used"):
    values = group["cvr"].dropna().values
    if len(values) > 1:
        channel_groups.append(values)

if len(channel_groups) >= 2:
    h_stat, h_p = stats.kruskal(*channel_groups)
else:
    h_stat, h_p = np.nan, np.nan

rho, rho_p = stats.spearmanr(attribution["click_share"], attribution["return_share"], nan_policy="omit")

stats_summary = pd.DataFrame(
    {
        "test": [
            "Kruskal-Wallis: channel CVR separation",
            "Spearman: click share vs return share",
        ],
        "statistic": [h_stat, rho],
        "p_value": [h_p, rho_p],
    }
)

display(stats_summary.round(4))


,test,statistic,p_value
0,Kruskal-Wallis: channel CVR separation,2.5133,0.7745
1,Spearman: click share vs return share,0.8286,0.0416


## Key Findings

Summarize the channel contribution differences that matter for budget decisions and for how performance reporting should be interpreted.


In [12]:
top_click = attribution.sort_values("click_share", ascending=False).iloc[0]
top_conversion = attribution.sort_values("conversion_share", ascending=False).iloc[0]
top_return = attribution.sort_values("return_share", ascending=False).iloc[0]
top_over = attribution.sort_values("possible_over_attribution_score", ascending=False).iloc[0]
top_under = attribution.sort_values("possible_under_attribution_score", ascending=False).iloc[0]
best_cvr = attribution.sort_values("cvr", ascending=False).iloc[0]
best_ros = attribution.sort_values("return_on_spend_proxy", ascending=False).iloc[0]

findings = []
findings.append(f"- Largest click contributor: {top_click['channel_used']} with {top_click['click_share']:.1%} of clicks.")
findings.append(f"- Largest conversion contributor: {top_conversion['channel_used']} with {top_conversion['conversion_share']:.1%} of estimated conversions.")
findings.append(f"- Largest return contributor: {top_return['channel_used']} with {top_return['return_share']:.1%} of estimated return proxy.")
findings.append(f"- Best conversion quality: {best_cvr['channel_used']} with CVR {best_cvr['cvr']:.1%}.")
findings.append(f"- Best return efficiency: {best_ros['channel_used']} with return-on-spend proxy {best_ros['return_on_spend_proxy']:.2f}.")
findings.append(
    f"- Strongest possible over-attribution pattern: {top_over['channel_used']} because click share ({top_over['click_share']:.1%}) exceeds conversion share ({top_over['conversion_share']:.1%}) and return share ({top_over['return_share']:.1%})."
)
findings.append(
    f"- Strongest possible under-attribution pattern: {top_under['channel_used']} because conversion share ({top_under['conversion_share']:.1%}) and return share ({top_under['return_share']:.1%}) exceed click share ({top_under['click_share']:.1%})."
)
if pd.notna(rho):
    findings.append(f"- Click share and return share have Spearman correlation {rho:.3f}; this helps judge whether attention and value contribution move together across channels.")
if pd.notna(h_p):
    findings.append(f"- Channel-level CVR differences produce p-value {h_p:.4f}, which helps indicate whether channels are meaningfully separated on conversion behavior.")

print("\n".join(findings))


- Largest click contributor: Email with 16.8% of clicks.
- Largest conversion contributor: Email with 16.9% of estimated conversions.
- Largest return contributor: Email with 16.8% of estimated return proxy.
- Best conversion quality: Email with CVR 8.0%.
- Best return efficiency: Facebook with return-on-spend proxy 5.03.
- Strongest possible over-attribution pattern: Instagram because click share (16.7%) exceeds conversion share (16.6%) and return share (16.6%).
- Strongest possible under-attribution pattern: Email because conversion share (16.9%) and return share (16.8%) exceed click share (16.8%).
- Click share and return share have Spearman correlation 0.829; this helps judge whether attention and value contribution move together across channels.
- Channel-level CVR differences produce p-value 0.7745, which helps indicate whether channels are meaningfully separated on conversion behavior.


## Strategic Interpretation

Budget strategy:

- Increase trust in channels that outperform on conversion share or return share relative to click share.
- Be cautious with channels that dominate clicks but lag in lower-funnel contribution, especially if they also consume disproportionate spend.
- Use segment-level channel results to avoid cutting channels that are weak on average but strong in specific customer segments.

Attribution strategy:

- Do not read click share as equivalent to business contribution.
- Pair top-of-funnel metrics with conversion-share and return-share diagnostics in monthly reporting.
- Treat the over-/under-attribution flags here as decision-support signals, not as a replacement for true multi-touch attribution.

Next best measurement upgrade:

- Add user-level touch sequences or assisted-conversion logs so future analyses can distinguish initiator channels from closer channels.
